###MCQ Creator Project 📝📝

In [ ]:
# Install the PDF reader and LangChain tools
!pip install -q pypdf langchain-community langchain-text-splitters langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.7/506.7 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.8 MB/s eta 0:00:00
ERROR: Operation cancelled by user


In [ ]:
#first we load the pdf
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = PyPDFLoader("/content/Water_Pollution_The_Problems_and_Solutions.pdf")
pages = loader.load()


In [ ]:
#we split the data
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    add_start_index=True

)

chunks = text_splitter.split_documents(pages)

In [ ]:
# Install the HuggingFace library
!pip install -q sentence-transformers
!pip install docarray

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 4.5 MB/s eta 0:00:00


In [ ]:
#we embed the data
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import DocArrayInMemorySearch
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

#we create the database
vectorstore=DocArrayInMemorySearch.from_documents(chunks, embeddings)

# now we create the retriever aka search engine

retriver = vectorstore.as_retriever()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
!pip install -q langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.2 MB/s eta 0:00:00


In [ ]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get('RealEst')
llm = ChatGroq(
    temperature = 0 ,
    model_name="llama-3.1-8b-instant" , # Updated to a currently supported model

)

In [ ]:
import random
random_chunk = random.choice(chunks).page_content

# 3. Create the Quiz Prompt
template = """
You are a helpful Personal Tutor. Based ONLY on the following context from a PDF,
create one Multiple Choice Question to test the student.
Provide 3 options (A, B, C) and indicate which one is correct.

CONTEXT:
{context}
"""
prompt = ChatPromptTemplate.from_template(template)

# 4. Generate the Question
chain = prompt | llm
response = chain.invoke({"context": random_chunk})

print("--- 📝 YOUR QUIZ QUESTION ---")
print(response.content)

--- 📝 YOUR QUIZ QUESTION ---
Here's a multiple choice question based on the context:

Question: What type of method has experienced a surge in popularity in recent years for removing water pollution?

A) Chemical treatment methods
B) Biological treatment methods
C) Physical treatment methods

Correct answer: B) Biological treatment methods


In [ ]:
import time

def run_tutor_session(num_questions=3):
    print(f"🎓 Starting a {num_questions}-question study session on Water Pollution...\n")

    for i in range(num_questions):
        # 1. Pick a random chunk
        chunk = random.choice(chunks).page_content

        # 2. Ask Groq to generate the question AND the answer key
        quiz_prompt = f"""
        Based on this text, create a multiple choice question with options A, B, and C.
        Then, at the very bottom, write 'Correct Answer: [Letter]'.

        TEXT: {chunk}
        """

        response = llm.invoke(quiz_prompt).content

        # Split the response to hide the answer from the user initially
        parts = response.split("Correct Answer:")
        question_text = parts[0]
        correct_answer = parts[1].strip() if len(parts) > 1 else "Unknown"

        print(f"--- Question {i+1} ---")
        print(question_text)

        # 3. Get User Input
        user_choice = input("Your Answer (A, B, or C): ").upper()

        # 4. Give Feedback
        if user_choice == correct_answer:
            print("🌟 CORRECT! You're mastering this.\n")
        else:
            print(f"❌ Not quite. The correct answer was {correct_answer}.\n")

        time.sleep(1) # Small pause for dramatic effect

# Start the session!
run_tutor_session(num_questions=3)

🎓 Starting a 3-question study session on Water Pollution...

--- Question 1 ---
What is a major challenge in addressing nonpoint source pollution?

A) Its limited geographical scope
B) Its multifaceted origins and elusive traceability
C) Its high cost of remediation


Your Answer (A, B, or C): B
🌟 CORRECT! You're mastering this.

--- Question 2 ---
Here's a multiple choice question based on the text:

What is a key factor in decreasing water pollution?

A) Raising awareness about the significance of conserving water sources
B) Using advanced technologies and wastewater treatment systems
C) Both A and B

D) None of the above


Your Answer (A, B, or C): C
🌟 CORRECT! You're mastering this.

--- Question 3 ---
Here's a multiple choice question based on the text:

What is required to reduce water pollution at its source?

A) Only the implementation of stormwater management
B) Stringent regulations on industrial discharges and agricultural effluent
C) The use of only prevention techniques




In [ ]:
import random
import time

def start_advanced_tutor(num_questions=5):
    score = 0
    review_list = []

    print("🎓 WELCOME TO YOUR PERSONAL AI ACADEMY")

    print("--------------------------------------------------\n")

    for i in range(num_questions):
        # 1. Select a chunk and identify its source page
        target_chunk = random.choice(chunks)
        context = target_chunk.page_content
        page_num = target_chunk.metadata.get('page', 'Unknown')

        # 2. Generate Question with Groq
        prompt = f"""
        Create a high-quality multiple choice question based on the text below.
        Format:
        Question: [Your Question]
        A) [Option]
        B) [Option]
        C) [Option]
        Correct Answer: [Single Letter]
        Explanation: [One sentence explaining why]

        TEXT: {context}
        """

        response = llm.invoke(prompt).content

        # 3. Parse the Response
        try:
            # We split the response to handle the interactive part
            main_body = response.split("Correct Answer:")[0]
            answer_part = response.split("Correct Answer:")[1]
            correct_letter = answer_part.strip()[0].upper()
            explanation = answer_part.split("Explanation:")[1].strip()
        except:
            continue # Skip if the AI messes up the format

        print(f"📝 QUESTION {i+1} (Source: Page {page_num+1})")
        print(main_body)

        user_answer = input("Your Answer (A, B, or C): ").strip().upper()

        # 4. Scoring and Feedback
        if user_answer == correct_letter:
            print("✅ EXCELLENT! That is correct.")
            score += 1
        else:
            print(f"❌ NOT QUITE. The correct answer was {correct_letter}.")
            print(f"💡 WHY: {explanation}")
            # Save for review
            review_list.append({"q": main_body, "a": correct_letter, "p": page_num+1})

        print("-" * 30 + "\n")
        time.sleep(0.5)

    # 5. THE GRADUATION REPORT
    print("📊 SESSION COMPLETE")
    print(f"Final Score: {score}/{num_questions} ({(score/num_questions)*100:.0f}%)")

    if review_list:
        print("\n📚 TOPICS TO REVIEW:")
        for item in review_list:
            print(f"- Study Page {item['p']} again regarding: {item['q'].split('?')[0]}?")
    else:
        print("\n🏆 PERFECT SCORE! You have mastered these sections.")

# Run it!
start_advanced_tutor(num_questions=3)

🎓 WELCOME TO YOUR PERSONAL AI ACADEMY
--------------------------------------------------

📝 QUESTION 1 (Source: Page 4)
Question: What is crucial for effective water pollution management and the provision of clean and safe water for future generations?

A) Individual efforts by communities alone
B) Collaboration of environmental organizations, government agencies, industries, and communities
C) Strict enforcement of policies without community involvement
D) Government regulations only


Your Answer (A, B, or C): B
✅ EXCELLENT! That is correct.
------------------------------

📝 QUESTION 2 (Source: Page 3)
Question: What is the purpose of introducing microorganisms or bacteria into polluted water in the process of bioremediation?
A) To increase the water's pH level
B) To purify the water of contaminants
C) To alter the water's temperature

Your Answer (A, B, or C): B
✅ EXCELLENT! That is correct.
------------------------------

📝 QUESTION 3 (Source: Page 2)
Question: What is a potential 

###News Analyst Project 📉📈📊

In [ ]:
# 1. LangChain & Groq (Core Logic)
!pip install -q langchain-groq langchain-core pydantic

# 2. Data Handling (For organizing the news tables)
!pip install -q pandas

# 3. HTTP Requests (To fetch news from the web or APIs)
!pip install -q requests

# 4. Optional: BeautifulSoup (If you want to scrape news sites directly later)
!pip install -q beautifulsoup4


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.8 MB/s eta 0:00:00


In [ ]:
from typing import List, Optional
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
import os
from google.colab import userdata

In [ ]:
# 1. Setup Groq
os.environ["GROQ_API_KEY"] = userdata.get('RealEst')
llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0)

In [ ]:
from typing import List, Optional
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
import os
from google.colab import userdata

# 2. Define the News Schema
class NewsAnalysis(BaseModel):
    category: str = Field(description="Politics, Economy, Security, or General")
    impact_score: str = Field(description="Scale of 1-10 on how much this affects daily life, represented as a string")
    one_sentence_summary: str = Field(description="A very brief summary of the event")
    personal_action: str = Field(description="What should the user do? (e.g., Stay home, withdraw cash, no action)")

# 3. Create the Structured LLM
structured_llm = llm.with_structured_output(NewsAnalysis)

In [ ]:
headlines = [
    "The Ministry of Energy announces a new tariff for private generators.",
    "Protests reported blocking the main highway in Jal el Dib.",
    "Lebanon wins a friendly basketball match against Jordan."
]

print("🗞️ ANALYZING THE MORNING NEWS...\n")

for news in headlines:
    analysis = structured_llm.invoke(f"Analyze this news for a resident in Lebanon: {news}")

    print(f"🔹 Headline: {news}")
    print(f"   [Category: {analysis.category} | Impact: {analysis.impact_score}/10]")
    print(f"   📝 Summary: {analysis.one_sentence_summary}")
    print(f"   🏃 Action: {analysis.personal_action}")
    print("-" * 50)

🗞️ ANALYZING THE MORNING NEWS...

🔹 Headline: The Ministry of Energy announces a new tariff for private generators.
   [Category: Economy | Impact: 8/10]
   📝 Summary: The Ministry of Energy has introduced a new tariff for private generators in Lebanon
   🏃 Action: Review your energy budget and consider adjusting your energy consumption habits
--------------------------------------------------
🔹 Headline: Protests reported blocking the main highway in Jal el Dib.
   [Category: Security | Impact: 6/10]
   📝 Summary: Protests are blocking the main highway in Jal el Dib, Lebanon
   🏃 Action: Avoid traveling to Jal el Dib and check for updates before using the highway
--------------------------------------------------
🔹 Headline: Lebanon wins a friendly basketball match against Jordan.
   [Category: General | Impact: 1/10]
   📝 Summary: Lebanon won a friendly basketball match against Jordan
   🏃 Action: no action
--------------------------------------------------


In [ ]:
import pandas as pd

# 1. Our list of "Dirty" data (Imagine these coming from a Telegram scraper)
raw_news_feed = [
    "Banque du Liban announces new circular regarding withdrawal limits for October.",
    "Major traffic jam on the Dora highway due to a broken down truck.",
    "Cloudy weather expected over the weekend with a slight drop in temperature.",
    "Reports of a fire in an industrial warehouse near the Beirut Port.",
    "Local bakery in Hamra offers free bread to those in need today."
]

def analyze_news_feed(feed):
    results = []

    print("🤖 Processing news through the AI Filter...")

    for item in feed:
        # The AI uses the schema we defined earlier to 'force' the output format
        analysis = structured_llm.invoke(f"Context: Lebanon. Analyze this: {item}")

        # We convert the Pydantic object into a simple Dictionary for the Table
        results.append({
            "Original Headline": item,
            "Category": analysis.category,
            "Impact": int(analysis.impact_score), # Convert to integer here
            "Summary": analysis.one_sentence_summary,
            "What to do": analysis.personal_action
        })

    # 2. Turn the list of dictionaries into a beautiful Table
    df = pd.DataFrame(results)
    return df

# Run the engine
news_table = analyze_news_feed(raw_news_feed)

# 3. Sort by Impact (Highest first) so the most important news is at the top
news_table = news_table.sort_values(by="Impact", ascending=False)

# Display the result
news_table

🤖 Processing news through the AI Filter...


,Original Headline,Category,Impact,Summary,What to do
0,Banque du Liban announces new circular regardi...,Economy,8,Banque du Liban has announced new withdrawal l...,"Review your finances and plan accordingly, con..."
1,Major traffic jam on the Dora highway due to a...,General,6,A broken down truck has caused a major traffic...,Consider alternative routes or leave early to ...
3,Reports of a fire in an industrial warehouse n...,Security,6,A fire has broken out in an industrial warehou...,Avoid the area and follow local news for updates
2,Cloudy weather expected over the weekend with ...,General,2,Lebanon is expected to experience cloudy weath...,No action needed
4,Local bakery in Hamra offers free bread to tho...,General,2,"A local bakery in Hamra, Lebanon is providing ...","No action required, but consider visiting the ..."


In [ ]:
def get_emergency_alerts(df, threshold=7):
    # Filter the table for high-impact news
    alerts = df[df['Impact'] >= threshold]

    if alerts.empty:
        print("✅ No critical alerts at the moment. It's a quiet day!")
    else:
        print(f"🚨 ATTENTION: Found {len(alerts)} high-impact events!\n")
        for index, row in alerts.iterrows():
            print(f"⚠️ {row['Category'].upper()} ALERT: {row['Summary']}")
            print(f"👉 Recommended Action: {row['What to do']}")
            print("-" * 30)

# Test our filter
get_emergency_alerts(news_table)

🚨 ATTENTION: Found 1 high-impact events!

⚠️ ECONOMY ALERT: Banque du Liban has announced new withdrawal limits for October, affecting citizens' access to their funds
👉 Recommended Action: Review your finances and plan accordingly, considering the new withdrawal limits
------------------------------
